# Test pipeline on Colab to leverage CUDA GPU


## Setup Repo

- The project has previously been imported via github, and the data folder with additional files uploaded manually
- The project is located under `drive/MyDive/project/ms-project`


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
%cd drive/MyDrive/project/ms-project

## Install dependencies

The original repo is using uv, but it is not suited to run on Colab with the Cuda devices so we install dependencies with pip instead


In [ ]:
!pip install torch
!pip install numpy
!pip install pillow
!pip install tqdm
!pip install colpali-engine
!pip install python-dotenv
!pip install pdf2image
!pip install datasets
!pip install mteb
!pip install ir-measures
!pip install einops
!pip install transformers
!pip install ollama
!pip install aiohttp
!pip install psutil
!pip install colab-xterm

## Run Ollama for the generation model

Ollama needs to run for the generation model qwen2.5vl:7b to be used, the following commands need to be entered in the terminal created below.

- `curl https://ollama.ai/install.sh | sh`
- `ollama serve &`
- `ollama pull qwen2.5vl:7b`


In [ ]:
%load_ext colabxterm
%xterm

## Reload imports

If a py file is edited, it won't import the revised version but the cache one.

This script enforces an import reload


In [ ]:
import importlib
import sys


def recursive_reload(package_name):
    """Recursively reload all modules in a given package. Useful for Colab or Jupyter after editing .py files."""
    modules_to_reload = [name for name in sys.modules if name.startswith(package_name)]

    for module_name in sorted(modules_to_reload, key=len, reverse=True):
        importlib.reload(sys.modules[module_name])
        print(f"Reloaded: {module_name}")

## Load the test script


In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

# Add src path for imports
src_path = Path.cwd() / "src"
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

load_dotenv()

# Load and reload with recursive call
from pipeline.rags.factory_rag import RAGFactory

recursive_reload("pipeline")
from pipeline.rags.factory_rag import RAGFactory


async def main():
    """Main function demonstrating the usage of MultiModalRAG."""
    # Configuration
    data_dir = os.getenv("RAGS_DATA_DIR")
    if data_dir is None:
        # Fallback to default if environment variable is not set
        data_dir = str(src_path / "data/rags")
        print(f"RAGS_DATA_DIR not set, using default: {data_dir}")

    data_dir = Path(data_dir)

    # Device configuration options (now passed through configs):
    # preferred_device = None  # Auto-detect best option
    # preferred_device = "cpu"  # Force CPU (most stable)
    # preferred_device = "mps"      # Force Apple Silicon GPU
    preferred_device = "cuda"  # Force NVIDIA GPU

    # Create configuration for the RAG factory
    config = {
        "type": "multimodal",
        "name": "multimodal_page",
        "configs": {
            "embedding_model": "colqwen2_embed",
            "generation_model": "colqwen2_ollama_gen",
            "preferred_device": preferred_device,
            "batch_size": 8,
            "chunking_strategy": "page",
            "knowledge_base": "consulting_light",
        },
    }

    # Initialize the RAG using the factory
    rag = RAGFactory.create_rag(config, data_dir)

    try:
        print("Starting indexing...")
        await rag.index()
        print("Indexing completed!")

        # Test answer generation
        test_query = "How have consumer shopping behaviors shifted towards online and mobile platforms since the onset of the COVID-19 pandemic?"
        answer, results = await rag.answer(test_query)
        print(f"Question:\n{test_query}")
        print("Retrieved documents:")
        for md, sc in results:
            print(f"  doc {md['corpus-id']}_{md['doc-id']} - score {sc}")

        print(
            f"Answer:\n{answer}",
        )

    except (FileNotFoundError, ValueError, RuntimeError) as e:
        print(f"Error occurred: {e}")
        import traceback

        traceback.print_exc()


## Run


In [ ]:
await main()